# Forecasting Jerárquico
### Bottom-up, top-down y reconciliación MinT

`Fase 3 · Video 16` — serie **Forecasting de Ventas de la A a la Z**

Hasta ahora pronosticamos SKUs por separado. Pero en Supply Chain los pronósticos de SKU, categoría y
total **deben cuadrar** — y casi nunca lo hacen. Aquí la reconciliación que los vuelve coherentes.

---
## 1. Librerías y carga de datos

In [ ]:
import warnings, sys
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from pathlib import Path

plt.rcParams.update({
    'figure.facecolor': '#F8FAFC', 'axes.facecolor': '#F8FAFC',
    'axes.grid': True, 'grid.color': '#E2E8F0',
    'grid.linewidth': 0.8, 'font.size': 11,
})
BLUE, RED, GREEN, PURPLE, ORANGE = '#2563EB','#DC2626','#16A34A','#7C3AED','#EA580C'
unit_fmt = FuncFormatter(lambda x, _: f'{x/1e3:.0f}k')
print('✅ Librerías cargadas')

In [ ]:
def find_csv(filename='sales_history.csv', max_levels=4):
    base = Path().resolve()
    for _ in range(max_levels):
        candidate = base / 'output' / filename
        if candidate.exists():
            return candidate
        base = base.parent
    raise FileNotFoundError(f"No se encontró '{filename}'. Corre main.py primero.")

csv_path = find_csv()
sys.path.insert(0, str(csv_path.parents[1]))
from src.hierarchy import summing_matrix, reconcile
from src.metrics import mase

df = pd.read_csv(csv_path, parse_dates=['date']).sort_values('date')
catalog = pd.read_csv(find_csv('sku_catalog.csv')).set_index('sku_id')
category_of = catalog['category'].to_dict()

# Trabajamos mensual (la reconciliación en Supply Chain suele ser mensual / S&OP)
mth = (df.groupby(['sku_id', pd.Grouper(key='date', freq='ME')])['units_sold']
         .sum().unstack('sku_id').fillna(0))
bottom = list(mth.columns)

S, labels = summing_matrix(bottom, category_of)
Y = S @ mth.T.values                       # (n_nodos × T): todas las series de la jerarquía
print(f'✅ Jerarquía: 1 Total + {catalog["category"].nunique()} categorías + {len(bottom)} SKUs = {len(labels)} nodos')
print(f'   {mth.shape[0]} meses de historia')

---
## 2. La jerarquía y la matriz de suma S

Una jerarquía se codifica en la **matriz de suma** `S`: mapea el nivel base (SKUs) a *todos* los nodos.
Si `b` es el vector de demanda de los SKUs, entonces `y = S · b` da la demanda de Total, categorías y SKUs
a la vez. **Cualquier** pronóstico coherente debe poder escribirse como `S · (algo a nivel SKU)`.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(S, aspect='auto', cmap='Blues')
ax.set_title('Matriz de suma S (filas = nodos, columnas = SKUs)', fontsize=13, fontweight='bold')
ax.set_ylabel('nodos: Total, categorías, SKUs →')
ax.set_xlabel('SKUs (nivel base)')
plt.colorbar(im, ax=ax, fraction=0.02)
plt.tight_layout()
plt.show()
print('Fila 0 (Total) = todo 1s (suma todos los SKUs). Filas de categoría = 1 en sus SKUs.')
print('Las últimas filas (SKUs) forman la identidad. Esa estructura es la que impone coherencia.')

---
## 3. El problema: los pronósticos base no cuadran

Generamos un pronóstico **independiente** por nodo (un Holt-Winters para el Total, otro por categoría, otro
por SKU). Al ser modelos distintos, **no son coherentes**: el pronóstico del Total no coincide con la suma
de los pronósticos de SKU.

In [ ]:
H, m = 12, 12
Ytr, Yte = Y[:, :-H], Y[:, -H:]

def ets_forecast(y):
    y = np.asarray(y, dtype=float)
    try:
        f = ExponentialSmoothing(np.maximum(y, 0.1), trend='add', seasonal='add',
                                 seasonal_periods=m, initialization_method='estimated'
                                 ).fit().forecast(H)
        return np.clip(f, 0, None)
    except Exception:
        return np.full(H, y[-m:].mean())

base = np.vstack([ets_forecast(Ytr[i]) for i in range(Y.shape[0])])   # incoherente

n_bottom = len(bottom)
total_base = base[0]
sum_skus_base = base[-n_bottom:].sum(axis=0)
incoh = np.abs(total_base - sum_skus_base).sum() / np.abs(total_base).sum()

fig, ax = plt.subplots(figsize=(13, 5))
h_idx = range(H)
ax.plot(h_idx, total_base, color=BLUE, linewidth=2.5, marker='o', label='Pronóstico del Total (modelo propio)')
ax.plot(h_idx, sum_skus_base, color=RED, linewidth=2.5, marker='s', label='Suma de los pronósticos de SKU')
ax.fill_between(h_idx, total_base, sum_skus_base, color=RED, alpha=0.12)
ax.yaxis.set_major_formatter(unit_fmt)
ax.set_title(f'Los pronósticos base NO cuadran (incoherencia media: {incoh:.1%})',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Mes del horizonte'); ax.legend(loc='upper left')
plt.tight_layout()
plt.show()
print(f'Incoherencia: |Total − Σ SKUs| / Total = {incoh:.1%}. Inaceptable para planear inventario.')

---
## 4. Reconciliación: bottom-up, top-down y MinT

Todas fuerzan coherencia proyectando los pronósticos base sobre el espacio coherente (`ỹ = S · G · ŷ`):

| Método | Idea | Cuándo brilla |
|---|---|---|
| **Bottom-up** | Suma los pronósticos de SKU hacia arriba | El detalle contiene la señal |
| **Top-down** | Reparte el Total por proporciones históricas | El agregado es más predecible |
| **MinT-OLS** | Combinación óptima con `W = I` | Teóricamente óptimo… si los errores son homogéneos |
| **MinT-WLS** | Combinación óptima ponderando por varianza del error | Robusto en la práctica |

MinT (Minimum Trace, Hyndman et al.) usa **todos los niveles a la vez** para minimizar el error total.

In [ ]:
# Varianzas del error base (in-sample, paso estacional) para WLS
resid_var = np.var(np.vstack([Ytr[i, m:] - Ytr[i, :-m] for i in range(Y.shape[0])]), axis=1) + 1e-6

methods = {
    'Base (incoherente)': base,
    'Bottom-up':          reconcile(base, S, 'bottom_up'),
    'MinT-OLS':           reconcile(base, S, 'ols'),
    'MinT-WLS':           reconcile(base, S, 'wls', resid_var),
}

cat_rows = [i for i, l in enumerate(labels) if l.startswith('cat:')]
sku_rows = list(range(len(labels) - n_bottom, len(labels)))

def level_mase(F, rows):
    return float(np.mean([mase(Yte[i], F[i], Ytr[i], m) for i in rows]))

rows = []
for name, F in methods.items():
    rows.append({'método': name,
                 'Total': mase(Yte[0], F[0], Ytr[0], m),
                 'Categoría': level_mase(F, cat_rows),
                 'SKU': level_mase(F, sku_rows)})
report = pd.DataFrame(rows).set_index('método')

fig, ax = plt.subplots(figsize=(12, 5))
report.plot(kind='bar', ax=ax, color=[BLUE, ORANGE, GREEN])
ax.axhline(1.0, color='black', linestyle='--', linewidth=1, label='MASE = 1 (seasonal naive)')
ax.set_ylabel('MASE (menor = mejor)')
ax.set_title('Precisión por nivel jerárquico', fontsize=13, fontweight='bold')
ax.legend(); plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

print(report.round(2).to_string())

---
## 5. Veredicto por nivel

In [ ]:
def incoherence(F):
    return np.abs(F[0] - F[-n_bottom:].sum(axis=0)).sum() / np.abs(F[0]).sum()

print('Coherencia (|Total − Σ SKUs| / Total):')
for name, F in methods.items():
    print(f'  {name:<20}: {incoherence(F):.3%}')

print('\nLectura honesta:')
best = report.mean(axis=1).idxmin()
print(f'  • TODAS las reconciliaciones dan coherencia ~0% (garantizada por construcción).')
print(f'    Ese es el beneficio seguro: los números cuadran entre niveles.')
print(f'  • En precisión, aquí gana {best}: agregar los pronósticos de SKU (que traen la señal fina)')
print(f'    corrige a los modelos independientes de Total/Categoría, que sobreajustan con poca historia.')
print(f'  • MinT-OLS puede ser INESTABLE (asume errores homogéneos); MinT-WLS, al ponderar por varianza,')
print(f'    es mucho más robusto. En la práctica: usa WLS, no OLS.')
print(f'  • No hay método universal: depende de en qué nivel vive tu señal. Mídelo, como siempre.')

---
## 6. Conclusiones y puente al Video 17

### Las reglas que te llevas

1. **La coherencia no es opcional** en Supply Chain: los pronósticos de SKU, categoría y total deben sumar.
   Modelos independientes por nivel **no cuadran** — y eso rompe la confianza del negocio.
2. **La reconciliación garantiza coherencia** proyectando sobre el espacio `S`. Ese beneficio es seguro.
3. **Bottom-up** es un default fuerte cuando el detalle trae la señal; **top-down** cuando el agregado es
   más predecible.
4. **MinT combina todos los niveles**, pero **OLS es frágil**: usa **WLS** (ponderado por varianza).
5. **Mide** qué nivel es más forecastable para elegir el método — no lo asumas.

### Cierre de los modelos clásicos y de ML

Ya recorrimos baselines, ARIMA, Prophet, intermitentes, ML y jerárquico. Falta mirar al futuro: los
**foundation models** que pronostican sin entrenar.

---

### Próximo video
**Video 17 — Foundation Models: TimeGPT, Chronos y el futuro del forecasting**
Zero-shot forecasting y una comparación honesta contra todo lo que construimos.